# Kolmogorov flow / 2D Navier-Stokes velocity dataset

This notebook generates `10_000` fluid-like velocity-field snapshots. The solver integrates forced 2D incompressible Navier-Stokes in vorticity form on a `256x256` grid, average-pools the recovered velocity channels `(u_x, u_y)` to `64x64`, and visualizes derived vorticity. This matches the paper's fine-grid simulation -> coarse state idea more closely.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess

REPO_URL = "https://github.com/SadreevAmir/DL_final_project.git"
REPO_DIR = "kolmogorov_dataset_repo"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
from google.colab import files
from IPython.display import display
from PIL import Image

from kolmogorov_dataset import KolmogorovConfig, generate_dataset

output_dir = Path("data/kolmogorov_velocity_256_to_64_10k")

config = KolmogorovConfig(
    output_dir=str(output_dir),
    total_images=10_000,
    grid_size=256,
    save_grid_size=64,
    num_trajectories=500,
    max_trajectories=2_000,
    snapshots_per_trajectory=20,
    burn_in_steps=5_000,
    save_interval=20,
    chunk_size=1_000,
    sim_batch_size=16,
    dt=0.005,
    viscosity=3.0e-4,
    drag=0.025,
    forcing_amp=0.55,
    forcing_mode=4,
    param_mode="mixed",
    initial_amplitude=1.5,
    spectral_filter_scale=12.0,
    output_field="velocity",
    normalize_output=False,
    velocity_scale=1.0,
    vorticity_scale=6.0,
    vorticity_clip=0.0,
    dtype="float32",
    compress=False,
    seed=123,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_threads=12,
    save_previews=True,
    preview_every_chunks=1,
    preview_count=32,
    save_sequence_previews=True,
    sequence_preview_count=16,
    min_image_std=0.03,
    min_image_range=0.25,
)

def show_and_download_preview(path: Path):
    print(f"Preview: {path}")
    display(Image.open(path))
    files.download(str(path))

def download_chunk(path: Path):
    print(f"Chunk: {path}")
    files.download(str(path))

paths = generate_dataset(
    config,
    on_chunk_saved=download_chunk,
    on_preview_saved=show_and_download_preview,
)

print(f"Saved {len(paths)} chunks")
files.download(str(output_dir / "manifest.json"))

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

manifest = json.loads((output_dir / "manifest.json").read_text())
print(json.dumps({
    "total_images": manifest["total_images"],
    "accepted_images": manifest["accepted_images"],
    "rejected_images": manifest["rejected_images"],
    "simulated_trajectories": manifest["simulated_trajectories"],
    "elapsed_seconds": manifest["elapsed_seconds"],
    "simulation_grid_size": manifest["simulation_grid_size"],
    "save_grid_size": manifest["save_grid_size"],
    "coarsening": manifest["coarsening"],
    "shape": manifest["shape"],
}, indent=2))

chunk = np.load(paths[0])
images = chunk["images"]
print("First chunk images:", images.shape, images.dtype, float(images.min()), float(images.max()))

if images.shape[1] == 2:
    ux = images[:32, 0]
    uy = images[:32, 1]
    # Approximate vorticity for visual inspection: d uy / dx - d ux / dy.
    visual = 0.5 * (np.roll(uy, -1, axis=-1) - np.roll(uy, 1, axis=-1)) - 0.5 * (np.roll(ux, -1, axis=-2) - np.roll(ux, 1, axis=-2))
else:
    visual = images[:32, 0]

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for ax, img in zip(axes.ravel(), visual):
    ax.imshow(img, cmap="RdBu_r")
    ax.axis("off")
plt.tight_layout()

In [ ]:
archive_path = "kolmogorov_velocity_64_10k_dataset.zip"
!zip -q -r {archive_path} {output_dir}
files.download(archive_path)